# Lab 2 — Global Post-hoc Explainability


---

This lab covers three **global post-hoc** explainability techniques applied to black-box classifiers on the Titanic dataset:

| Technique | What it explains |
|---|---|
| **Permutation Feature Importance** | How much each feature contributes to model performance |
| **Partial Dependence Plot (PDP)** | Marginal effect of one (or two) features on the predicted outcome |
| **Global Surrogate Models** | A white-box model that approximates a black-box model's predictions |

**Guide:**
- ⚙️ Data preprocessing (Titanic)
- 🔀 Permutation Feature Importance
- 📉 Partial Dependence Plots
- 🪞 Global Surrogate Models


## ⚙️ Data Preprocessing (~10 min)

We are using the **Titanic** dataset loaded directly from OpenML.
This section walks through the full preprocessing pipeline: loading, splitting, imputing, encoding, and scaling.

> 💡 Apply the same good habits from Lab 1: always fit transformations on the **training set only**, then apply them to the test set!


### ⚙️ Step 1 — Imports & Load

In [ ]:
# Import the required libraries for this exercise

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn import tree

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

### ⚙️ Step 2 — Load the Titanic Dataset

In [ ]:
# Load input features and target variable
df, y = fetch_openml("titanic", version=1, as_frame=True, parser='auto', return_X_y=True)

# The "survived" column contains the target variable
df["survived"] = y

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1309, 14)


,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,survived
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO",1
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON",1
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",0
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON",0
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",0


In [ ]:
# Quick overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   name       1309 non-null   object  
 2   sex        1309 non-null   category
 3   age        1046 non-null   float64 
 4   sibsp      1309 non-null   int64   
 5   parch      1309 non-null   int64   
 6   ticket     1309 non-null   object  
 7   fare       1308 non-null   float64 
 8   cabin      295 non-null    object  
 9   embarked   1307 non-null   category
 10  boat       486 non-null    object  
 11  body       121 non-null    float64 
 12  home.dest  745 non-null    object  
 13  survived   1309 non-null   category
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB


In [ ]:
# Class balance — important for choosing metrics!
print("Class distribution:")
print(df["survived"].value_counts())
print(f"\nImbalance ratio: {df['survived'].value_counts()[0]/df['survived'].value_counts()[1]:.2f}:1")

Class distribution:
survived
0    809
1    500
Name: count, dtype: int64

Imbalance ratio: 1.62:1


/tmp/ipykernel_558/912207974.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"\nImbalance ratio: {df['survived'].value_counts()[0]/df['survived'].value_counts()[1]:.2f}:1")
/tmp/ipykernel_558/912207974.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"\nImbalance ratio: {df['survived'].value_counts()[0]/df['survived'].value_counts()[1]:.2f}:1")


### ⚙️ Step 3 — Stratified Train / Test Split

> 🔑 **Key reminder:** split *before* any transformation. Fitting scalers or imputers on the full dataset causes **data leakage**.


In [ ]:
# Split the dataset. 80% for training data and 20% for test data. Shuffle the dataset and perform stratification by label
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=True, random_state=42, stratify=df['survived'])


print(f"Train: {len(df_train)} samples | Test: {len(df_test)} samples")
print("\nTrain class distribution:")
print(df_train["survived"].value_counts())

Train: 1047 samples | Test: 262 samples

Train class distribution:
survived
0    647
1    400
Name: count, dtype: int64


### ⚙️ Step 4 — Handle Missing Values

> 🔑 **Key reminder:** compute statistics (mean, median…) on the **training set only**, then apply to both sets.


In [ ]:
#### START CODE HERE #### (~5-6 lines of code)
# Count NaN per column and print
nan_train = df_train.isna().sum()
nan_test  = df_test.isna().sum()

print("Missing values — Train:")
print(nan_train[nan_train > 0])
print("\nMissing values — Test:")
print(nan_test[nan_test > 0])

#### END CODE HERE ####

Missing values — Train:
age          209
fare           1
cabin        822
boat         658
body         955
home.dest    450
dtype: int64

Missing values — Test:
age           54
cabin        192
embarked       2
boat         165
body         233
home.dest    114
dtype: int64


In [ ]:
print(f'Number of null values in Train before pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

df_train['age'] = df_train['age'].fillna(df_train['age'].mean())
df_test['age'] = df_test['age'].fillna(df_train['age'].mean())

print(f'Number of null values in Train after pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

Number of null values in Train before pre-processing: 209/1047
Number of null values in Test before pre-processing: 54/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262


In [ ]:
print(f'Number of null values in Train before pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

df_train['fare'] = df_train['fare'].fillna(df_train['fare'].median())
df_test['fare'] = df_test['fare'].fillna(df_train['fare'].median())

print(f'Number of null values in Train after pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

Number of null values in Train before pre-processing: 1/1047
Number of null values in Test before pre-processing: 0/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262


In [ ]:
print(f'Number of null values in Train before pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

imp = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df_train[['embarked']] = imp.fit_transform(df_train[['embarked']])
df_test[['embarked']] = imp.transform(df_test[['embarked']])

print(f'Number of null values in Train after pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

Number of null values in Train before pre-processing: 0/1047
Number of null values in Test before pre-processing: 2/262
Number of null values in Train after pre-processing: 0/1047
Number of null values in Test after pre-processing: 0/262


### ⚙️ Step 5 — Drop Uninformative Columns

> 🔑 Remove columns that are not useful for predicting survival, or that leak information about the target variable.


In [ ]:
df_train = df_train.drop(columns=['name','ticket'])
df_test = df_test.drop(columns=['name','ticket'])

df_train.head()

,pclass,sex,age,sibsp,parch,fare,cabin,embarked,boat,body,home.dest,survived
999,3,female,29.604316,0,0,7.7500,NaN,Q,15 16,NaN,NaN,1
392,2,female,24.000000,1,0,27.7208,NaN,C,12,NaN,"Lucca, Italy / California",1
628,3,female,11.000000,4,2,31.2750,NaN,S,NaN,NaN,"Sweden Winnipeg, MN",0
1165,3,male,25.000000,0,0,7.2250,NaN,C,NaN,NaN,NaN,0
604,3,female,16.000000,0,0,7.6500,NaN,S,16,NaN,"Norway Los Angeles, CA",1


In [ ]:
# Remove columns that contain target-leaking info (boat/body) or are mostly missing (cabin, home.dest)
df_train = df_train.drop(columns=['cabin', 'body', 'boat', 'home.dest'])
df_test = df_test.drop(columns=['cabin', 'body', 'boat', 'home.dest'])

### ⚙️ Step 6 — Extract Features & Apply Encoding / Scaling

> 🔑 **Key reminder:** ML models need numbers. We use `OneHotEncoder` for categoricals and `MinMaxScaler` for numerics — both fitted on **train only**.


In [ ]:
# Extract target variable and input features for the training data
y_train = df_train['survived']               # Target variable training set
X_train = df_train.drop('survived', axis=1)  # Features training set

# Extract target variable and input features for the testing data
y_test = df_test['survived']                 # Target variable test set
X_test = df_test.drop('survived', axis=1)    # Features test set

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

categorical_columns = ["pclass", "sex", "embarked"]
numerical_columns = ["sibsp", "parch", "fare", "age"]

categorical_encoder = OneHotEncoder(handle_unknown='ignore', drop='if_binary')
minmax_s = MinMaxScaler()

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", minmax_s, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

> ✅ **Preprocessing complete.** Remember the three rules:
> 1. Split *before* fitting any transformer.
> 2. Fit transformers on **train only**, apply to both.
> 3. Use `ColumnTransformer` to cleanly handle mixed-type features in a `Pipeline`.


## Exercise 1 — Permutation Feature Importance 🔀

**Permutation Feature Importance** is a model-agnostic inspection technique that measures how much each feature contributes to model performance.

> A feature is **important** if shuffling its values causes the *model error to increase*. If the model relied on that feature, disrupting it should hurt performance.

**Key advantages:**
- Direct and intuitive interpretation of the model's behaviour
- Fully **model-agnostic** — works with any fitted estimator
- Does **not require retraining** the model

**Main disadvantages:**
- Assumes **feature independence** — correlated features can produce biased importance estimates
- Tightly coupled to the chosen **performance metric**

---

### Your tasks
1. Fit a [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) on the Titanic dataset and inspect its **accuracy** on train and test.

### Exercise 1.1 — Feature Importances
2. Calculate **built-in feature importances** (as in Lab 1).
> * What is the most important feature? Is it categorical or numerical?
> * You can aggregate (via sum) importances of dummy columns for the same original categorical attribute.

### Exercise 1.2 — Permutation Importance
3. Compute [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html).
> * First on the **test set**. What are the most important features?
> * Repeat on the **training set**. Did you get the same result? If not, what does this suggest?

### Exercise 1.3 — Regularisation vs Overfitting
4. Instantiate a second `RandomForestClassifier` with `min_samples_leaf=20`.
> * Repeat permutation importance on both train and test. Has anything changed?
> * What does this tell you about Random Forest overfitting and the `fare` feature?

> ***Hint:*** In the first case, the discrepancy between train and test permutation importance can be explained by overfitting — the RF uses `fare` to memorise training examples.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_classification

### 1.1 — Fit the Random Forest Classifier

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 1.2 — Model Performance


In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 1.3 — Built-in Feature Importances

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 1.4 — Permutation Importance (Test)

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 1.5 — Permutation Importance (Train)

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 1.6 — Regularised Random Forest (`min_samples_leaf=20`)

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

## Exercise 2 — Partial Dependence Plots 📉

A **Partial Dependence Plot** (PDP) shows the **marginal effect** of one (or two) features on the predicted outcome of a machine learning model.

- It is a **global method**: it considers *all instances* and describes the global relationship between a feature and the prediction.
- PDPs are easy to visualise and intuitively interpretable.

**Key advantages:**
- Computation is intuitive and the visualisation is easy to read.
- Provides a global view of how a feature influences the prediction.

**Main disadvantages:**
- Assumes **feature independence** — correlated features distort the marginal estimates.
- Only practical for **at most two features** at a time.

---

### Your tasks
1. Instantiate and fit a [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) and a [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) on the Titanic dataset.
2. **Evaluate** both models — inspect accuracy on train and test.
3. Print **feature importances** for both models.
4. Generate **PDPs** using [`PartialDependenceDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html) for both models.


### 2.1 — Imports

In [ ]:
from sklearn.inspection import permutation_importance, partial_dependence, PartialDependenceDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification

### 2.2 — Fit RF & DT Classifiers

In [ ]:
#### START CODE HERE -- Random Forest ####


#### END CODE HERE ####

In [ ]:
#### START CODE HERE -- Decision Tree ####


#### END CODE HERE ####

### 2.3 — Feature Importances

In [ ]:
#### START CODE HERE -- Feature Importance RF ####


#### END CODE HERE ####

In [ ]:
#### START CODE HERE -- Feature Importance DT ####


#### END CODE HERE ####

### 2.4 — Partial Dependence Plots

In [ ]:
#### START CODE HERE -- PDP Random Forest ####


#### END CODE HERE ####

In [ ]:
#### START CODE HERE -- PDP Decision Tree ####


#### END CODE HERE ####

## Exercise 3 — Global Surrogate Models 🪞

A **global surrogate model** is an *interpretable* model trained to **approximate** the predictions of a black-box model as accurately as possible.

> The key idea: you don't need to peek inside the black box — only its **prediction function** and a dataset are required.

**How to build a surrogate:**
1. Select a dataset X.
2. Get the **black-box predictions** for X.
3. Choose an **interpretable model type** (linear model, decision tree, …).
4. Train the interpretable model on X using the black-box predictions as labels.
5. Measure how well the surrogate **replicates** the black-box predictions (reconstruction error).

The surrogate is **model-agnostic** — it makes no assumptions about the inner workings of the black-box model.

---

### Your tasks
1. Instantiate and fit a **black-box model**: [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) on the Titanic dataset.

2. Instantiate and fit a **white-box surrogate**: [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) trained on X and the *black-box predictions*.

3. **Evaluate** both models:
> * Calculate `accuracy_score` for both on train and test.
> * Calculate the **reconstruction error** — the loss of fidelity when using black-box predictions to supervise the white-box model.

4. **Inspect the surrogate's coefficients.**

> ***Hint:*** The reconstruction error measures how faithfully the white-box model mimics the black-box — lower error means a more faithful surrogate.


### 3.1 — Imports

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### 3.2 — Train the Black-Box Model

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 3.3 — Train the White-Box Surrogate

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 3.4 — Inspect Surrogate Coefficients

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 3.5 — Evaluate Both Models

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 3.6 — Reconstruction Error

In [ ]:
#### START CODE HERE ####


#### END CODE HERE ####

### 🔍 Reflection Questions

1. **Permutation Feature Importance:** Why does permutation importance on the *training set* give a different result than on the *test set* for the unconstrained Random Forest? What does `min_samples_leaf=20` change, and why?

2. **PDP:** The PDP shows the *average* marginal effect. If `fare` and `pclass` are correlated, what risk do you run when interpreting each PDP independently? How could you detect this problem?

3. **Surrogate Model:** A high reconstruction error means the white-box model cannot replicate the black-box well. In that case, is the surrogate still a valid explanation of the black-box? What would you do?

4. **General:** All three methods are **post-hoc** and **global**. Can a global explanation mislead you in a specific individual prediction? Give an example using the Titanic context (e.g. first-class women vs third-class men).
